In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from beta_evidence_field import (OccupancyField,
                                 OccupancyObservation)

from tools.plotting import configure_plots


configure_plots()

# Computational Scaling

The computational cost of a Beta Evidence Field depends primarily on the number of retained observations \(N\) and the number of queried spatial locations \(M\).

Two operations dominate the current implementation:

1. observation-state redundancy is evaluated from pairwise similarities between retained observations;
2. occupied and free-space evidence is evaluated between the retained observations and requested query locations.

The following experiments characterise the computational scaling of these operations and establish a baseline for subsequent optimisation.

In [ ]:
def benchmark(function,
              repeats : int = 10) -> float:

    function()

    times = []

    for _ in range(repeats):

        start = time.perf_counter()

        function()

        times.append(time.perf_counter() - start)

    return float(np.median(times))

In [ ]:
def observation_set(n):

    y = np.linspace(0.0, 1.0, n)

    return [OccupancyObservation(origin    = np.array([0.0, y_i]),
                                 direction = np.array([1.0, 0.0]),
                                 range     = 1.0,
                                 returned  = True,
                                 time      = 0.0)
            for y_i in y]

## Redundancy Computation

Sampling-density normalisation requires the local observation-state density

$$
\rho_i
=
\sum_j
K_s(s_i,s_j),
$$

from which the redundancy weight is

$$
w_i=\rho_i^{-1}.
$$

The current implementation evaluates the complete pairwise state-distance matrix before truncating similarities outside the configured state support. The resulting computation therefore remains quadratic in the number of retained observations,

$$
T_{\mathrm{red}}(N)=O(N^2).
$$

This experiment measures that scaling directly.

In [ ]:
sample_counts = np.unique(
    np.geomspace(10,
                 3000,
                 15,
                 dtype = int)
)

redundancy_times = []


for n in sample_counts:

    field        = OccupancyField.from_yaml("../config/example_2d_config.yaml")
    observations = observation_set(n)

    runtime = benchmark(
        lambda: field._redundancy_weights(observations),
        repeats = 5)

    redundancy_times.append(runtime)


redundancy_times = np.asarray(redundancy_times)

In [ ]:
quadratic_reference = redundancy_times[0] * (
    sample_counts/sample_counts[0]
)**2


fig, ax = plt.subplots()

ax.loglog(sample_counts,
          redundancy_times,
          "o-",
          color = "tab:orange",
          alpha = 0.7,
          label = "Measured")

ax.loglog(sample_counts,
          quadratic_reference,
          "--",
          color = "0.5",
          alpha = 0.7,
          label = r"$O(N^2)$")

ax.set_xlabel("Number of retained observations")
ax.set_ylabel("Redundancy computation time [s]")

ax.legend()

plt.show()

## Query Scaling with Retained Observations

Once redundancy weights have been cached, querying the current field does not require the observation-state similarity matrix to be recomputed.

For \(N\) retained observations and \(M\) query locations, the dominant query operation evaluates spatial evidence between observations and locations. With \(M\) fixed, query cost should therefore increase approximately linearly with the number of retained observations.

In [ ]:
query_count = 500

y = np.linspace(-0.5,
                1.5,
                query_count)

points = np.column_stack((np.ones_like(y),
                          y))

sample_counts = np.unique(
    np.geomspace(10,
                 2000,
                 15,
                 dtype = int)
)

query_times = []


for n in sample_counts:

    field = OccupancyField.from_yaml("../config/example_2d_config.yaml")
    field.add(observation_set(n))

    runtime = benchmark(
        lambda: field.query(points),
        repeats = 10)

    query_times.append(runtime)


query_times = np.asarray(query_times)

In [ ]:
linear_reference = query_times[0] * (
    sample_counts/sample_counts[0]
)


fig, ax = plt.subplots()

ax.loglog(sample_counts,
          query_times,
          "o-",
          color = "tab:blue",
          alpha = 0.7,
          label = "Measured")

ax.loglog(sample_counts,
          linear_reference,
          "--",
          color = "0.5",
          alpha = 0.7,
          label = r"$O(N)$")

ax.set_xlabel("Number of retained observations")
ax.set_ylabel("Query time [s]")

ax.legend()

plt.show()

## Query Scaling with Spatial Resolution

The field representation is independent of the locations at which it is queried. Increasing the number of query locations therefore increases output resolution without changing the underlying field.

With the retained observation set fixed, the computational cost should increase approximately linearly with the number of query locations \(M\).

In [ ]:
field = OccupancyField.from_yaml("../config/example_2d_config.yaml")
field.add(observation_set(500))

query_counts = np.unique(
    np.geomspace(10,
                 10000,
                 15,
                 dtype = int)
)

query_times = []


for m in query_counts:

    y = np.linspace(-0.5,
                    1.5,
                    m)

    points = np.column_stack((np.ones_like(y),
                              y))

    runtime = benchmark(
        lambda: field.query(points),
        repeats = 10)

    query_times.append(runtime)


query_times = np.asarray(query_times)

In [ ]:
linear_reference = query_times[0] * (
    query_counts/query_counts[0]
)


fig, ax = plt.subplots()

ax.loglog(query_counts,
          query_times,
          "o-",
          color = "tab:blue",
          alpha = 0.7,
          label = "Measured")

ax.loglog(query_counts,
          linear_reference,
          "--",
          color = "0.5",
          alpha = 0.7,
          label = r"$O(M)$")

ax.set_xlabel("Number of query locations")
ax.set_ylabel("Query time [s]")

ax.legend()

plt.show()

## Redundancy Caching

Observation redundancy depends on the retained observation set rather than on the spatial locations being queried.

The redundancy weights are therefore computed when the retained observation set changes and cached for subsequent current-field queries. This avoids repeatedly evaluating the same pairwise state similarities when the same field is queried at different spatial locations.

The following experiment compares the cost of a normal cached query with the cost of explicitly recomputing redundancy before every query.

In [ ]:
sample_counts = np.unique(
    np.geomspace(10,
                 2000,
                 15,
                 dtype = int)
)

y = np.linspace(-0.5,
                1.5,
                500)

points = np.column_stack((np.ones_like(y),
                          y))

cached_times     = []
recomputed_times = []


for n in sample_counts:

    field = OccupancyField.from_yaml("../config/example_2d_config.yaml")
    field.add(observation_set(n))

    cached = benchmark(
        lambda: field.query(points),
        repeats = 10)

    def recomputed_query():

        field._redundancy = field._redundancy_weights(field.observations)

        return field.query(points)

    recomputed = benchmark(
        recomputed_query,
        repeats = 5)

    cached_times.append(cached)
    recomputed_times.append(recomputed)


cached_times     = np.asarray(cached_times)
recomputed_times = np.asarray(recomputed_times)

In [ ]:
fig, ax = plt.subplots()

ax.loglog(sample_counts,
          recomputed_times,
          "o-",
          color = "tab:orange",
          alpha = 0.7,
          label = "Recompute redundancy")

ax.loglog(sample_counts,
          cached_times,
          "o-",
          color = "tab:blue",
          alpha = 0.7,
          label = "Cached redundancy")

ax.set_xlabel("Number of retained observations")
ax.set_ylabel("Query time [s]")

ax.legend()

plt.show()

## Summary

The computational experiments characterise the current dense implementation and establish a baseline for subsequent optimisation.

Redundancy evaluation is performed when the retained observation set changes, while cached redundancy weights are reused for subsequent spatial queries. Query cost is determined primarily by the number of retained observations and requested query locations.

The current redundancy implementation still constructs dense pairwise state-distance and similarity matrices. Finite state support removes negligible interactions from the resulting density estimate but does not avoid constructing these matrices. Sparse state-neighbour evaluation can therefore target this remaining cost without changing the underlying sampling-density normalisation.